In [ ]:
# import torch
# torch.__version__

In [ ]:
!pip install torch_geometric
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.4.0+cu121.html

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.9 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.4.0+cu121.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 14.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 99.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 105.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 89.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 986.2/986.2 kB 57.6 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv  # Import GCNConv
from sklearn.metrics import f1_score  # To calculate F1 score

# Load the Cora dataset
dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]

# Define the MLP for edge probability
class EdgeProbMLP(nn.Module):
    def __init__(self, in_channels, hidden_dim):
        super(EdgeProbMLP, self).__init__()
        self.fc1 = nn.Linear(2 * in_channels, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, node_features, edge_index):
        edge_features = torch.cat([node_features[edge_index[0]], node_features[edge_index[1]]], dim=1)
        x = F.relu(self.fc1(edge_features))
        prob = torch.sigmoid(self.fc2(x))
        return prob

# Define the overall model with Edge Probabilities and GCNConv (GNN)
class GNNModel(nn.Module):
    def __init__(self, in_channels, hidden_dim, out_channels, num_classes):
        super(GNNModel, self).__init__()
        self.edge_prob_mlp = EdgeProbMLP(in_channels, hidden_dim)
        self.gcn1 = GCNConv(in_channels, hidden_dim)
        self.gcn2 = GCNConv(hidden_dim, num_classes)

    def forward(self, data, edge_index):
        # Forward pass through two layers of GCNConv with the given edges
        x = F.relu(self.gcn1(data.x, edge_index))
        out = self.gcn2(x, edge_index)
        return out

# Reparameterization trick for edge sampling
def reparameterization_trick(edge_probs, edge_index, q=500):
    eps = torch.randn_like(edge_probs)
    z = edge_probs + eps * torch.sqrt(edge_probs * (1 - edge_probs))  # Reparameterization trick
    z = torch.clamp(z, min=0, max=1)  # Make sure the sampled probabilities are in [0, 1]
    sampled_edges = torch.multinomial(z, q, replacement=False)
    return sampled_edges

# F1 score calculation helper
def calculate_f1(logits, labels, mask):
    preds = logits[mask].argmax(dim=1)
    f1 = f1_score(labels[mask].cpu(), preds.cpu(), average='micro')
    return f1

# Training function (uses sampled edges)
def train(model, data, optimizer, criterion, q=500):
    model.train()
    optimizer.zero_grad()

    # Compute edge probabilities
    edge_probs = model.edge_prob_mlp(data.x, data.edge_index).squeeze()

    # Sample edges using the reparameterization trick
    sampled_edge_indices = reparameterization_trick(edge_probs, data.edge_index, q=q)
    sampled_edge_index = data.edge_index[:, sampled_edge_indices]

    # Forward pass with sampled edges
    out = model(data, sampled_edge_index)

    # Compute node classification loss
    loss = criterion(out[data.train_mask], data.y[data.train_mask])

    # Backpropagate loss
    loss.backward()

    # Accumulate gradients and update parameters
    optimizer.step()

    return loss.item()

# Evaluation function (uses full graph)
def evaluate(model, data):
    model.eval()
    with torch.no_grad():
        # Use the full graph's edge_index for evaluation (no sampling)
        out = model(data, data.edge_index)

    # Calculate F1 scores for train, val, and test sets
    train_f1 = calculate_f1(out, data.y, data.train_mask)
    val_f1 = calculate_f1(out, data.y, data.val_mask)
    test_f1 = calculate_f1(out, data.y, data.test_mask)

    return train_f1, val_f1, test_f1



Processing...
Done!


In [ ]:
# Main training loop
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GNNModel(in_channels=dataset.num_node_features, hidden_dim=128, out_channels=64, num_classes=dataset.num_classes).to(device)
data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

for epoch in range(200):
    loss = train(model, data, optimizer, criterion, q=1000)
    train_f1, val_f1, test_f1 = evaluate(model, data)

    print(f'Epoch {epoch}, Loss: {loss:.4f}, Train F1: {train_f1:.4f}, Val F1: {val_f1:.4f}, Test F1: {test_f1:.4f}')

Epoch 0, Loss: 1.9364, Train F1: 0.9500, Val F1: 0.6720, Test F1: 0.7060
Epoch 1, Loss: 1.4792, Train F1: 0.9714, Val F1: 0.7120, Test F1: 0.7350
Epoch 2, Loss: 1.0301, Train F1: 0.9714, Val F1: 0.7420, Test F1: 0.7510
Epoch 3, Loss: 0.6927, Train F1: 0.9643, Val F1: 0.7420, Test F1: 0.7650
Epoch 4, Loss: 0.4493, Train F1: 0.9714, Val F1: 0.7500, Test F1: 0.7640
Epoch 5, Loss: 0.2174, Train F1: 0.9714, Val F1: 0.7360, Test F1: 0.7610
Epoch 6, Loss: 0.2279, Train F1: 0.9714, Val F1: 0.7560, Test F1: 0.7590
Epoch 7, Loss: 0.1615, Train F1: 0.9786, Val F1: 0.7660, Test F1: 0.7810
Epoch 8, Loss: 0.1196, Train F1: 0.9786, Val F1: 0.7600, Test F1: 0.7840
Epoch 9, Loss: 0.0600, Train F1: 0.9929, Val F1: 0.7600, Test F1: 0.7830
Epoch 10, Loss: 0.0299, Train F1: 0.9929, Val F1: 0.7580, Test F1: 0.7870
Epoch 11, Loss: 0.0322, Train F1: 0.9929, Val F1: 0.7580, Test F1: 0.7890
Epoch 12, Loss: 0.0340, Train F1: 0.9857, Val F1: 0.7640, Test F1: 0.7910
Epoch 13, Loss: 0.1019, Train F1: 0.9857, Val F1

In [ ]:
data

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])

In [ ]:
sum(data.train_mask)

tensor(140, device='cuda:0')

In [ ]:
140/2708

0.051698670605613